### NUEVA RUTINA GENERAR DATOS

Rutina Multiparamétrica para filtrar UNIVERSO de datos a FORECASTEAR

In [1]:
# Importar librerías estándar
import os
import shutil
import time
from datetime import datetime
from random import randint

# Importar librerías de terceros
import pandas as pd
import numpy as np
from dotenv import dotenv_values

# Importar funciones necesarias del módulo `funciones_forecast`
# from forecast_core.funciones_forecast import (
#     Open_Conn_Postgres,
#     Open_Connection,
#     Close_Connection,
#     Open_Diarco_Data,
#     get_execution_by_status,
#     Open_Postgres_retry,
#     mover_archivos_procesados,
#     actualizar_site_ids,
#     insertar_graficos_forecast,
#     get_precios,
#     get_execution_execute_by_status,
#     update_execution,
#     update_execution_execute,
#     create_execution_execute_result,
#     generar_mini_grafico,
#     generar_grafico_base64
# )

# Importar librerías adicionales necesarias
import ace_tools_open as tools



In [2]:
import sys
import os
from dotenv import dotenv_values

ENV_PATH = os.environ.get("FORECAST_ENV_PATH", "C:/ETL/FORECAST/.env")  # Toma Producción si está definido, o la ruta por defecto
# Verificar si el archivo .env existe
if not os.path.exists(ENV_PATH):
    print(f"El archivo .env no existe en la ruta: {ENV_PATH}")
    print(f"Directorio actual: {os.getcwd()}")
    sys.exit(1)

secrets = dotenv_values(ENV_PATH)
folder = secrets["FOLDER_DATOS"]

print("Folder:", folder)


Folder: data


In [3]:
print(f"{secrets['BASE_DIR']}/{secrets['FOLDER_DATOS']}")


C:/ETL/FORECAST/data


In [ ]:
import pyodbc
print(pyodbc.version)

In [ ]:
fes = get_execution_execute_by_status(50)
tools.display_dataframe_to_user(name="Contenido de Archivos Markdown", dataframe=fes)

# RUTINA DE PRUEBA

In [4]:
# Acceso a Datos
from dotenv import dotenv_values
import psycopg2 as pg2
import pyodbc
import sqlalchemy
from sqlalchemy import text

In [6]:
def Open_Diarco_Data(): 
    conn_str = f"dbname={secrets['PG_DB']} user={secrets['PG_USER']} password={secrets['PG_PASSWORD']} host={secrets['PG_HOST']} port={secrets['PG_PORT']}"
    #print (conn_str)
    for i in range(5):
        try:    
            conn = pg2.connect(conn_str)
            return conn
        except Exception as e:
            print(f'Error en la conexión: {e}')
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan

from sqlalchemy import create_engine
from sqlalchemy.exc import OperationalError
import time

def Open_Diarco_Data_SQLAlchemy():
    conn_str = (
        f"postgresql://{secrets['PG_USER']}:{secrets['PG_PASSWORD']}"
        f"@{secrets['PG_HOST']}:{secrets['PG_PORT']}/{secrets['PG_DB']}"
    )

    for i in range(5):
        try:
            engine = create_engine(conn_str, pool_pre_ping=True)
            # Probar la conexión
            with engine.connect() as connection:
                return engine  # Retorna el engine si la conexión fue exitosa
        except OperationalError as e:
            print(f'Error en la conexión: {e}')
            time.sleep(5)

    return None  # Retorna None si todos los intentos fallan

def Close_Connection(conn): 
    if conn is not None:
        conn.close()
        # print("✅ Conexión cerrada.")    
    return True

In [15]:
# Nueva Rutina al MIGRAR a PostgreSQL y Ejecución REMOTA
# 2025-05-16 Se agrega c_comprador
def generar_datos(id_proveedor, etiqueta, sucursales, rubros):
    folder = secrets["FOLDER_DATOS"]
    folder = f'{secrets["BASE_DIR"]}/{folder}'  # Asegurarse de que la ruta sea absoluta
    archivo_datos = f'{folder}/{etiqueta}.csv'
    archivo_articulos = f'{folder}/{etiqueta}_articulos.csv'
    archivo_stock = f'{folder}/{etiqueta}_stock_sucursal.csv'


    
    # Verificar la fecha de modificación del archivo
    if os.path.exists(archivo_datos):
        fecha_modificacion = datetime.fromtimestamp(os.path.getmtime(archivo_datos))
        if fecha_modificacion.date() == datetime.today().date():
            try:
                data = pd.read_csv(archivo_datos)
                data = data.dropna(subset=['Codigo_Articulo', 'Sucursal', 'Fecha'], how='all')  # Eliminar filas donde todas las columnas clave son NaN
                data['Codigo_Articulo'] = data['Codigo_Articulo'].astype(int)
                data['Sucursal'] = data['Sucursal'].astype(int)
                data['Fecha'] = pd.to_datetime(data['Fecha'])

                articulos = pd.read_csv(archivo_articulos)
                articulos = articulos.dropna(subset=['C_ARTICULO', 'C_SUCU_EMPR'], how='all')  # Eliminar filas donde todas las columnas clave son NaN
                

                print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {etiqueta}")
                return data, articulos
            except Exception as e:
                print(f"Error al leer los archivos cacheados: {e}")
        else:
            # Eliminar archivos si la fecha no es la de hoy
            os.remove(archivo_datos)
            if os.path.exists(archivo_articulos):
                os.remove(archivo_articulos)
            print(f"-> Archivos eliminados por ser obsoletos: {archivo_datos}, {archivo_articulos}")
    else:
        print(f"-> Generando datos para ID: {id_proveedor}, Label: {etiqueta}")
        # Aquí puedes incluir el código para generar los datos si no EXISTE el Archivo en el CACHE
        conn = Open_Diarco_Data_SQLAlchemy()

        # --- ARTÍCULOS --- NUEVA FUENTE GLOBAL 06/25 -- SP_BASE_PRODUCTOS_VIGENTES
        if sucursales:  # Verificar si sucursales no está vacío
            query_articulos = f"""
                SELECT DISTINCT c_sucu_empr, c_articulo, c_proveedor_primario, abastecimiento, cod_cd, habilitado,  
                        cod_comprador AS c_comprador, 
                        q_factor_compra, full_capacity_pallet, number_of_layers, number_of_boxes_per_layer
                FROM src.base_productos_vigentes
                WHERE c_proveedor_primario = ({id_proveedor})
                        AND c_sucu_empr IN ({sucursales})
                ORDER BY c_articulo, c_proveedor_primario;  
            """
        else:
            query_articulos = f"""
                SELECT DISTINCT c_sucu_empr, c_articulo, c_proveedor_primario, abastecimiento, cod_cd, habilitado,  
                        cod_comprador AS c_comprador, 
                        q_factor_compra, full_capacity_pallet, number_of_layers, number_of_boxes_per_layer
                FROM src.base_productos_vigentes
                WHERE c_proveedor_primario = ({id_proveedor})
                ORDER BY c_articulo, c_proveedor_primario;  
            """
        articulos = pd.read_sql(query_articulos, conn) # type: ignore
        if articulos.empty:
            print(f"❗ No se encontraron artículos para el proveedor {id_proveedor}.")
            #Close_Connection(conn)
            return None, None

        articulos.columns = articulos.columns.str.upper()
        articulos.rename(columns={'COD_COMPRADOR': 'C_COMPRADOR'}, inplace=True)   # En la base nueva se llama distinto
        articulos['C_SUCU_EMPR'] = articulos['C_SUCU_EMPR'].astype(int)
        articulos['C_ARTICULO'] = articulos['C_ARTICULO'].astype(int)
        articulos['C_PROVEEDOR_PRIMARIO'] = articulos['C_PROVEEDOR_PRIMARIO'].astype(int)
        articulos['ABASTECIMIENTO'] = articulos['ABASTECIMIENTO'].astype(int)
        articulos['HABILITADO'] = articulos['HABILITADO'].astype(int)
        articulos['C_COMPRADOR'] = articulos['C_COMPRADOR'].astype(int)
        articulos['Q_FACTOR_COMPRA'] = articulos['Q_FACTOR_COMPRA'].astype(int)
        articulos['FULL_CAPACITY_PALLET'] = articulos['FULL_CAPACITY_PALLET'].astype(int)
        articulos['NUMBER_OF_LAYERS'] = articulos['NUMBER_OF_LAYERS'].astype(int)
        articulos['NUMBER_OF_BOXES_PER_LAYER'] = articulos['NUMBER_OF_BOXES_PER_LAYER'].astype(int)
        articulos.to_csv(archivo_articulos, index=False, encoding='utf-8')
        print(f"---> Datos de Artículos guardados")        
        
        # --- BASE STOCK --- NUEVA FUENTE GLOBAL 06/25 -- SP_BASE_STOCK_SUCURSAL
        if sucursales:  # Verificar si sucursales no está vacío
            query_stock_sucursal = f"""
                SELECT codigo_articulo, codigo_sucursal, codigo_proveedor, pedido_sgm, stock, 
                    pedido_pendiente, i_lista_calculado, factor_venta, precio_venta, precio_costo, 
                    q_dias_stock, transfer_pendiente, venta_unidades_1q, venta_unidades_2q
                FROM src.base_stock_sucursal
                WHERE codigo_proveedor = {id_proveedor}
                    AND codigo_sucursal IN ({sucursales})
                ORDER BY codigo_articulo, codigo_articulo;
            """
        else:
         # Si no hay sucursales, traer todas las sucursales del proveedor
            # Esto puede ser costoso si hay muchas sucursales, pero es necesario para evitar
            query_stock_sucursal = f"""
                SELECT codigo_articulo, codigo_sucursal, codigo_proveedor, pedido_sgm, stock, 
                    pedido_pendiente, i_lista_calculado, factor_venta, precio_venta, precio_costo, 
                    q_dias_stock, transfer_pendiente, venta_unidades_1q, venta_unidades_2q
                FROM src.base_stock_sucursal
                WHERE codigo_proveedor = {id_proveedor}
                ORDER BY codigo_articulo, codigo_articulo;
        """
        stock_sucursal = pd.read_sql(query_stock_sucursal, conn) # type: ignore
        if stock_sucursal.empty:
            print(f"❗ No se encontraron artículos de Stock_Sucursal para el proveedor {id_proveedor}.")
            #Close_Connection(conn)
            return None, None
        
        # Limpieza general antes de conversión
        stock_sucursal = stock_sucursal.replace([np.inf, -np.inf], np.nan)
        stock_sucursal = stock_sucursal.fillna(0)

        stock_sucursal.columns = stock_sucursal.columns.str.upper()
        #  Cambiar tipos de datos
        stock_sucursal['CODIGO_SUCURSAL'] = stock_sucursal['CODIGO_SUCURSAL'].astype(int)
        stock_sucursal['CODIGO_ARTICULO'] = stock_sucursal['CODIGO_ARTICULO'].astype(int)
        stock_sucursal['CODIGO_PROVEEDOR'] = stock_sucursal['CODIGO_PROVEEDOR'].astype(int)
        stock_sucursal["PEDIDO_SGM"] = pd.to_numeric(stock_sucursal["PEDIDO_SGM"], errors="coerce").astype("Float64")
        stock_sucursal["STOCK"] = pd.to_numeric(stock_sucursal["STOCK"], errors="coerce").astype("Float64")
        stock_sucursal["PEDIDO_PENDIENTE"] = pd.to_numeric(stock_sucursal["PEDIDO_PENDIENTE"], errors="coerce").astype("Float64")
        stock_sucursal["I_LISTA_CALCULADO"] = pd.to_numeric(stock_sucursal["I_LISTA_CALCULADO"], errors="coerce").astype("Float64")
        stock_sucursal['FACTOR_VENTA'] = stock_sucursal['FACTOR_VENTA'].astype(int)
        stock_sucursal['PRECIO_VENTA'] = pd.to_numeric(stock_sucursal['PRECIO_VENTA'], errors='coerce').astype('Float64')
        stock_sucursal['PRECIO_COSTO'] = pd.to_numeric(stock_sucursal['PRECIO_COSTO'], errors='coerce').astype('Float64')
        stock_sucursal['Q_DIAS_STOCK'] = pd.to_numeric(stock_sucursal['Q_DIAS_STOCK'], errors='coerce').astype('Int64')
        stock_sucursal['TRANSFER_PENDIENTE'] = pd.to_numeric(stock_sucursal['TRANSFER_PENDIENTE'], errors='coerce').astype('Float64')
        stock_sucursal['VENTA_UNIDADES_1Q'] = pd.to_numeric(stock_sucursal['VENTA_UNIDADES_1Q'], errors='coerce').astype('Float64')
        stock_sucursal['VENTA_UNIDADES_2Q'] = pd.to_numeric(stock_sucursal['VENTA_UNIDADES_2Q'], errors='coerce').astype('Float64')

        stock_sucursal.to_csv(archivo_stock, index=False, encoding='utf-8')
        print(f"---> Datos de Stock Sucursal guardados")
        
        # -- COMBINAR ARTÍCULOS y STOCK --
        articulos = pd.merge(articulos, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')  

        # --- VENTAS --- DIARCO + BARRIO ( En 2 Bases de Datos distintas )
        if sucursales:  # Verificar si sucursales no está vacío
            query_ventas = f"""
                SELECT 
                    v.f_venta AS Fecha, 
                    v.c_articulo AS Codigo_Articulo, 
                    v.c_sucu_empr AS Sucursal, 
                    v.q_unidades_vendidas AS Unidades
                FROM src.t702_est_vtas_por_articulo v
                JOIN src.base_productos_vigentes a 
                    ON a.c_articulo = v.c_articulo
                    AND a.c_sucu_empr = v.c_sucu_empr
                    AND a.c_proveedor_primario = {id_proveedor}
                WHERE v.f_venta >= '2024-06-01'
                    AND v.c_sucu_empr IN ({sucursales})

                UNION ALL

                SELECT 
                    v.f_venta AS Fecha, 
                    v.c_articulo AS Codigo_Articulo, 
                    v.c_sucu_empr AS Sucursal, 
                    v.q_unidades_vendidas AS Unidades
                FROM src.t702_est_vtas_por_articulo_dbarrio v
                JOIN src.base_productos_vigentes a 
                    ON a.c_articulo = v.c_articulo
                    AND a.c_sucu_empr = v.c_sucu_empr
                    AND a.c_proveedor_primario = {id_proveedor}
                WHERE v.f_venta >= '2024-06-01'
                    AND v.c_sucu_empr IN ({sucursales})

                ORDER BY Fecha, Codigo_Articulo, Sucursal;
            """
        else:   
            query_ventas = f"""
                SELECT 
                    v.f_venta AS Fecha, 
                    v.c_articulo AS Codigo_Articulo, 
                    v.c_sucu_empr AS Sucursal, 
                    v.q_unidades_vendidas AS Unidades
                FROM src.t702_est_vtas_por_articulo v
                JOIN src.base_productos_vigentes a 
                    ON a.c_articulo = v.c_articulo
                    AND a.c_sucu_empr = v.c_sucu_empr
                    AND a.c_proveedor_primario = {id_proveedor}
                WHERE v.f_venta >= '2024-06-01'

                UNION ALL

                SELECT 
                    v.f_venta AS Fecha, 
                    v.c_articulo AS Codigo_Articulo, 
                    v.c_sucu_empr AS Sucursal, 
                    v.q_unidades_vendidas AS Unidades
                FROM src.t702_est_vtas_por_articulo_dbarrio v
                JOIN src.base_productos_vigentes a 
                    ON a.c_articulo = v.c_articulo
                    AND a.c_sucu_empr = v.c_sucu_empr
                    AND a.c_proveedor_primario = {id_proveedor}
                WHERE v.f_venta >= '2024-06-01'

                ORDER BY Fecha, Codigo_Articulo, Sucursal;
            """
        ventas = pd.read_sql(query_ventas, conn) # type: ignore
        if ventas.empty:
            print(f"⚠️ No se encontraron ventas DIARCO + BARRIO para el proveedor {id_proveedor}.")
        
        # Convertir columnas a minúsculas si hay datos
        if not ventas.empty:
            ventas.columns = ventas.columns.str.lower()
            # Transformar tipos de datos si hay datos
            ventas['sucursal'] = ventas['sucursal'].astype(int)
            ventas['codigo_articulo'] = ventas['codigo_articulo'].astype(int)
            ventas['fecha'] = pd.to_datetime(ventas['fecha'])
            # Eliminar filas con NaN en 'fecha', 'codigo_articulo' o 'sucursal'
            ventas = ventas.dropna(subset=['fecha', 'codigo_articulo', 'sucursal'], how='all')
            
        else:
            print(f"⚠️ No se encontraron ventas para el proveedor {id_proveedor} ni en DIARCO ni en BARRIO.")
            ventas = pd.DataFrame(columns=['fecha', 'codigo_articulo', 'sucursal', 'unidades'])  # DataFrame vacío con columnas esperadas

        ventas = ventas.rename(columns={
            "fecha": "Fecha",
            "codigo_articulo": "Codigo_Articulo",
            "sucursal": "Sucursal",
            "unidades": "Unidades"
        })
        # Guardar solo VENTAS 
        ventas.to_csv(f'{folder}/{etiqueta}_Demanda.csv', index=False, encoding='utf-8')
        print(f"---> Datos de Ventas guardados")

        # --- MERGE ---
        data = pd.merge(
            articulos,
            ventas,  
            left_on=['C_ARTICULO', 'C_SUCU_EMPR'],          
            right_on=['Codigo_Articulo', 'Sucursal'],            
            how='left'   # 'inner'  # Solo traer los productos que están en AMBOS ARCHIVOS  'left' trate TODOS los articulos activos en cada sucursal
        )

        if data.empty:
            print(f"⚠️ No hay coincidencias entre artículos y ventas para el proveedor {id_proveedor}.")
            #Close_Connection(conn)
            return None, articulos

        # Conversión segura de columnas a enteros con soporte para NaN
        data['C_ARTICULO'] = data['C_ARTICULO'].astype('Int64')
        data['C_SUCU_EMPR'] = data['C_SUCU_EMPR'].astype('Int64')
        data['Codigo_Articulo'] = data['Codigo_Articulo'].astype('Int64')
        data['Sucursal'] = data['Sucursal'].astype('Int64')

        # Agregar columna indicadora de si tiene demanda
        data['TIENE_DEMANDA'] = data['Unidades'].notna().astype(int)
        data['Unidades'] = data['Unidades'].fillna(0)

        # Guardado
        data.to_csv(archivo_datos, index=False, encoding='utf-8')
        print(f"---> Datos de RECUPERACIÓN guardados")
        
        # Guardar los datos Compactos de VENTAS en un archivo CSV con el nombre del Proveedor y sufijo _Ventas
        file_path = f'{folder}/{etiqueta}_Ventas.csv'
        print(f"[DEBUG] Ruta destino definida en .env: {folder}")

        # Eliminar Columnas Innecesarias
        data = data[['Fecha', 'Codigo_Articulo', 'Sucursal', 'Unidades']]
        data.to_csv(file_path, index=False, encoding='utf-8')
        print(f"---> Datos de Ventas guardados: {file_path}")  

        #Close_Connection(conn)
        return data, articulos

In [18]:
id_proveedor = 275
etiqueta = f"{id_proveedor}_SWIFT_FULL"
sucursales = None
rubros = "'2250', '2245', '6628'"

In [19]:
data, articulos = generar_datos(id_proveedor=id_proveedor, etiqueta=etiqueta, sucursales=sucursales, rubros=rubros) # type: ignore

-> Generando datos para ID: 275, Label: 275_SWIFT_FULL
---> Datos de Artículos guardados
---> Datos de Stock Sucursal guardados
---> Datos de Ventas guardados
---> Datos de RECUPERACIÓN guardados
[DEBUG] Ruta destino definida en .env: C:/ETL/FORECAST/data
---> Datos de Ventas guardados: C:/ETL/FORECAST/data/275_SWIFT_FULL_Ventas.csv


In [ ]:

elegido = '596_PROCTER_ALGO_01'
for index, row in fes[fes["name"] == elegido].iterrows():
   
# for index, row in fes[fes["fee_status_id"] == 50].iterrows():
    algoritmo = row["name"]
    name = algoritmo.split('_ALGO')[0]
    execution_id = row["forecast_execution_id"]
    id_proveedor = row["ext_supplier_code"]
    forecast_execution_execute_id = row["forecast_execution_execute_id"]
    supplier_id = row["supplier_id"]

    folderP = folder + '/procesado'

    print(f"Algoritmo: {algoritmo}  - Name: {name} exce_id: {forecast_execution_execute_id} id: Proveedor {id_proveedor}")
    print(f"supplier-id: {supplier_id} ----------------------------------------------------")

    